# Pipeline de previsão de dengue

Orquestração do pipeline: o Airflow executa este notebook (papermill) com os parâmetros de cada tarefa, e ele chama o pacote `atlas_ml`, que tem o código de dados, janelas, modelos e avaliação.

| `step` | O que faz | Resultado (`result_path`) |
|---|---|---|
| `prepare` | Lê a gold do BigQuery (ou `painel_parquet`) e grava o snapshot numa run `dataset` | `dataset_run_id` |
| `train` | Treina `model` no `horizon` sobre `dataset_run`, avalia e registra no MLflow | `run_id`, `model_version`, `metrics` |
| `compare` | Junta as runs de `pipeline_run` e `horizon` e registra a run `compare` | `run_id` |

À mão: `papermill forecast_pipeline.ipynb saida.ipynb -p step train -p model catboost -p dataset_run <ID>`.

In [ ]:
# Parâmetros (o papermill injeta uma célula com os valores recebidos logo depois desta)
step = "train"  # prepare | train | compare
model = ""  # train: baseline_bayes, chronos, catboost, svr, lstm ou ensemble
horizon = 1  # semanas à frente (1 a 4); no compare, 0 compara todos
window = 12  # semanas de histórico por janela
seed = 42
features = ""  # vazio = canais do SINAN + relevo; "-" = só a incidência; ou lista com vírgulas
dataset_run = ""  # train: run `dataset` com o snapshot
base_runs = {}  # só ensemble: {"catboost": "<run_id>", "svr": "<run_id>", "lstm": "<run_id>"}
pipeline_run = "manual"  # tag atlas.pipeline_run (a DAG passa o run_id dela)
fonte = ""  # prepare: tabela projeto.dataset.tabela no lugar da gold do ambiente
painel_parquet = ""  # prepare: parquet com um painel pronto, em vez de ler o BigQuery
register = True  # train: registra atlas-br-<modelo>-h<horizonte>
result_path = ""  # arquivo JSON do resultado (o executor define; vazio = só imprime)
out_dir = ""  # diretório desta execução (o executor define)

In [ ]:
# Contrato com o pacote e validação dos parâmetros: falha antes de criar qualquer run
REQUIRES_API = 1  # versão mínima de atlas_ml.forecast.NOTEBOOK_API de que este notebook precisa

import json
import os
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from atlas_ml.forecast.notebook import notebook_sha, require_api
from atlas_ml.models import MODELS

require_api(REQUIRES_API)
STEPS = ("prepare", "train", "compare")
if step not in STEPS:
    raise ValueError(f"step inválido {step!r}: use {', '.join(STEPS)}")
if step == "train":
    if model not in MODELS:
        raise ValueError(f"model inválido {model!r}: use {', '.join(MODELS)}")
    if not dataset_run:
        raise ValueError("train precisa de dataset_run (a run `dataset` do prepare)")
    if model == "ensemble" and not base_runs:
        raise ValueError("ensemble precisa de base_runs (catboost, svr e lstm)")
result = {}

In [ ]:
# Ambiente: threads, credenciais/versões e MLflow (o que o antigo pipeline.py fazia)
from atlas_ml.forecast.threads import limitar

limitar()  # antes de importar PyTorch, CatBoost e afins: respeita ATLAS_THREADS

import mlflow

from atlas_ml.config import load_settings

settings = load_settings()
_ = settings.git_sha  # falha cedo se o commit da imagem não estiver definido
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
os.environ["ATLAS_PIPELINE_RUN"] = pipeline_run  # tag atlas.pipeline_run de todas as runs
os.environ["ATLAS_NOTEBOOK_SHA"] = notebook_sha()  # tag atlas.notebook_sha
print(
    f"step={step} model={model or '-'} horizon={horizon} pipeline_run={pipeline_run} "
    f"notebook={os.environ['ATLAS_NOTEBOOK_SHA']}"
)

In [ ]:
# prepare: snapshot dos dados numa run `dataset`
if step == "prepare":
    if painel_parquet:
        from atlas_ml.forecast.dataset import prepare_from_parquet

        dataset_run_id = prepare_from_parquet(settings, painel_parquet)
    else:
        from atlas_ml.data.bigquery import bigquery_client
        from atlas_ml.forecast.dataset import prepare

        dataset_run_id = prepare(settings, bigquery_client(settings), fonte or None)
    result = {"dataset_run_id": dataset_run_id}
    print(f"dataset_run_id={dataset_run_id}")

In [ ]:
# train: um modelo, um horizonte, sobre o snapshot
if step == "train":
    from atlas_ml.forecast.base import run_model
    from atlas_ml.forecast.config import ForecastConfig
    from atlas_ml.models import get_factory

    if not features.strip():
        cfg = ForecastConfig(window=int(window), horizon=int(horizon), seed=int(seed))
    else:
        lista = tuple(f.strip() for f in features.split(",") if f.strip() and f.strip() != "-")
        cfg = ForecastConfig(window=int(window), horizon=int(horizon), seed=int(seed), features=lista)
    kwargs = {"base_runs": dict(base_runs)} if model == "ensemble" else {}
    r = run_model(get_factory(model, **kwargs), cfg, dataset_run, settings, register=bool(register))
    metricas = {k: round(float(v), 6) for k, v in r.metrics.items() if k.startswith("test_")}
    result = {"run_id": r.run_id, "model_version": r.model_version, "metrics": metricas}
    print(" ".join(f"{k}={v:.4f}" for k, v in metricas.items()))
    print(f"run_id={r.run_id}")

In [ ]:
# compare: junta as runs de uma execução (e horizonte) numa run `compare`
if step == "compare":
    from atlas_ml.forecast.compare import compare

    run_id, tabela = compare(settings, pipeline_run, int(horizon) or None)
    result = {"run_id": run_id}
    print(tabela)
    print(f"run_id={run_id}")

In [ ]:
# Resultado do passo: o Airflow lê este arquivo para passar o run_id adiante
if result_path:
    Path(result_path).parent.mkdir(parents=True, exist_ok=True)
    Path(result_path).write_text(json.dumps(result, ensure_ascii=False), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False))